# PawSense breed classifier (Oxford-IIIT Pet, 37 breeds)

Runs the same steps as `train.py` (it imports its functions, so the two never drift).
Measured results are in `ml/README.md`.

Locally: open from `ml/` with the `.venv` kernel. In Colab: run the first cell.

In [ ]:
# Colab only: get the repo and install deps (local runs use ml/.venv instead).
import sys
if "google.colab" in sys.modules:
    !git clone -q https://github.com/elysian12/pawsense.git
    %cd pawsense/ml
    !pip -q install tensorflow-datasets coremltools

In [ ]:
import json, time
from train import build_datasets, build_model, train_head, fine_tune, evaluate_and_save

BACKBONE = "efficientnetb0"  # or "mobilenetv3small" (smaller, less accurate)
started = time.time()
train, val, test, names = build_datasets()  # train[:90%] / train[90%:] / test, via tensorflow_datasets
print(len(names), "classes:", names)

Model: ImageNet backbone (rescaling layer inside) + dropout + 37-way softmax. Input is raw RGB 0..255, 224x224.

In [ ]:
model, backbone = build_model(len(names), BACKBONE)
model.summary()

Stage 1: backbone frozen, train the head (Adam 1e-3).

In [ ]:
train_head(model, train, val)

Stage 2: unfreeze the top backbone layers (BatchNorm stays frozen), Adam 1e-4, early stopping on validation top-1.

In [ ]:
fine_tune(model, backbone, train, val)

Test split top-1 / top-5; saves `artifacts/breeds.keras`, `labels/breeds.txt`, `artifacts/metrics.json`.

In [ ]:
print(json.dumps(evaluate_and_save(model, test, names, BACKBONE, started), indent=2))

Exports. `export_coreml.py` needs macOS: Core ML prediction (used for its accuracy check) only runs there.

In [ ]:
!python export_tflite.py
!python export_coreml.py